In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import xgboost as xgb
from skopt import BayesSearchCV
from skopt.space import Real, Integer

In [2]:
data_prefix = "../0_data/processed_data/"
model_types = ['RF', 'XGB', 'LGB']
non_feature_cols = ['SMILES', 'MP', 'Type', 'MP']

In [ ]:

def model_development(data, non_feature_cols, model_type):

    X = data.drop(columns=non_feature_cols)
    y = data['MP'].values
    strat_labels = data['Ro5'].values

    # Precompute folds stratified on Ro5 label (not continuous MP)
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    folds = list(skf.split(X, strat_labels))

    # ── Helper: run 10-fold CV for a given model instance ─────────────
    def run_cv(model_instance):
        fold_rmses = []
        for train_idx, val_idx in folds:
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            model_instance.fit(X_train, y_train)
            preds = model_instance.predict(X_val)
            fold_rmses.append(np.sqrt(mean_squared_error(y_val, preds)))
        return fold_rmses

    # ── Model + search space definitions ──────────────────────────────
    if model_type == 'RF':
        default_model = RandomForestRegressor(random_state=42, n_jobs=-1)
        base_model    = RandomForestRegressor(random_state=42, n_jobs=-1)
        search_space  = {
            'n_estimators':      Integer(50, 500),
            'max_depth':         Integer(3,    20),
            'min_samples_split': Integer(2,    20),
            'min_samples_leaf':  Integer(1,    10),
            'max_features':      Real(0.1, 1.0),
        }

    elif model_type == 'LGB':
        default_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
        base_model    = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
        search_space  = {
            'num_leaves':        Integer(20,   300),
            'max_depth':         Integer(3,    12),
            'learning_rate':     Real(0.01, 0.3,   prior='log-uniform'),
            'n_estimators':      Integer(50, 500),
            'min_child_samples': Integer(10,   50),
            'subsample':         Real(0.6, 1.0),
            'colsample_bytree':  Real(0.4, 1.0),
            'reg_alpha':         Real(1e-5, 10.0,  prior='log-uniform'),
            'reg_lambda':        Real(1e-5, 10.0,  prior='log-uniform'),
        }

    elif model_type == 'XGB':
        default_model = xgb.XGBRegressor(random_state=42, n_jobs=-1,
                                          tree_method='hist', verbosity=0)
        base_model    = xgb.XGBRegressor(random_state=42, n_jobs=-1,
                                          tree_method='hist', verbosity=0)
        search_space  = {
            'n_estimators':      Integer(50, 500),
            'max_depth':         Integer(3,    12),
            'learning_rate':     Real(0.01, 0.3,  prior='log-uniform'),
            'subsample':         Real(0.6, 1.0),
            'colsample_bytree':  Real(0.4, 1.0),
            'reg_alpha':         Real(1e-5, 10.0, prior='log-uniform'),
            'reg_lambda':        Real(1e-5, 10.0, prior='log-uniform'),
            'min_child_weight':  Integer(1,   10),
        }

    else:
        raise ValueError(f"model_type must be 'RF', 'LGB', or 'XGB'; got '{model_type}'")

    # ── Trial 0: default hyperparameters ──────────────────────────────
    trial_results = {}
    fold_rmses_0 = run_cv(default_model)
    mean_0 = float(np.mean(fold_rmses_0))
    std_0  = float(np.std(fold_rmses_0))
    trial_results[0] = {'fold_rmses': fold_rmses_0, 'mean_rmse': mean_0, 'std_rmse': std_0}
    print(f"Trial  0 (default) | mean RMSE: {mean_0:.4f} ± {std_0:.4f}")

    # ── Trials 1-20: BayesSearchCV ────────────────────────────────────
    # Pass precomputed folds as cv so BayesSearchCV never calls
    # StratifiedKFold.split(X, y) with a continuous y (which would error).
    opt = BayesSearchCV(
        base_model,
        search_space,
        n_iter=20,
        cv=folds,
        scoring='neg_root_mean_squared_error',
        random_state=42,
        n_jobs=1,
        refit=True,
    )
    opt.fit(X, y)

    # Extract per-trial fold RMSEs from cv_results_ (negate the neg-RMSE scores)
    n_splits = len(folds)
    for i in range(20):
        fold_rmses = [-opt.cv_results_[f'split{s}_test_score'][i] for s in range(n_splits)]
        mean_rmse  = float(np.mean(fold_rmses))
        std_rmse   = float(np.std(fold_rmses))
        trial_results[i + 1] = {
            'fold_rmses': fold_rmses,
            'mean_rmse':  mean_rmse,
            'std_rmse':   std_rmse,
        }
        print(f"Trial {i+1:>2d} | mean RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}")

    # ── Best params & model (BayesSearchCV already refits on all data) ─
    best_model_info = {
        'best_params': opt.best_params_,
        'model':       opt.best_estimator_,
    }

    return trial_results, best_model_info


In [23]:
model_development_results_dict = {}
for model in model_types:

    data = pd.read_parquet(data_prefix + f"data_with_selected_features_{model}_scaled.parquet").head(300)
    print(f"\n=== Model: {model} ===")
    # print the number of data and features
    print(f"Dataset shape: {data.shape} (n_samples={data.shape[0]}, n_features={data.shape[1] - len(non_feature_cols)})")
    trial_results, best_model_info = model_development(data, non_feature_cols, model_type=model)
    model_development_results_dict[model] = {
        'trial_results': trial_results,
        'best_model_info': best_model_info,
    }



=== Model: RF ===
Data shape: (300, 55) (n_samples=300, n_features=51)


/Users/zeqing/opt/anaconda3/envs/LA/lib/python3.6/site-packages/sklearn/model_selection/_split.py:668: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  % (min_groups, self.n_splits)), UserWarning)


Trial  0 (default) | mean RMSE: 46.9669 ± 7.2581
Trial  1 | mean RMSE: 48.0623 ± 6.8437
Trial  2 | mean RMSE: 47.5112 ± 7.0291
Trial  3 | mean RMSE: 46.7248 ± 7.0750
Trial  4 | mean RMSE: 47.8359 ± 6.8138
Trial  5 | mean RMSE: 47.4594 ± 6.7406
Trial  6 | mean RMSE: 46.3587 ± 7.1237
Trial  7 | mean RMSE: 47.3827 ± 6.9117
Trial  8 | mean RMSE: 47.6332 ± 6.9973
Trial  9 | mean RMSE: 48.0062 ± 6.8329
Trial 10 | mean RMSE: 49.4854 ± 7.0036
Trial 11 | mean RMSE: 46.3532 ± 7.1677
Trial 12 | mean RMSE: 45.9145 ± 6.7332
Trial 13 | mean RMSE: 46.0234 ± 6.7876
Trial 14 | mean RMSE: 46.0876 ± 7.0792
Trial 15 | mean RMSE: 46.4887 ± 6.5196
Trial 16 | mean RMSE: 47.1574 ± 6.5899
Trial 17 | mean RMSE: 46.2491 ± 6.4798
Trial 18 | mean RMSE: 46.1033 ± 6.7525
Trial 19 | mean RMSE: 49.4357 ± 6.7488
Trial 20 | mean RMSE: 49.5001 ± 6.8576

=== Model: XGB ===
Data shape: (300, 24) (n_samples=300, n_features=20)


/Users/zeqing/opt/anaconda3/envs/LA/lib/python3.6/site-packages/sklearn/model_selection/_split.py:668: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  % (min_groups, self.n_splits)), UserWarning)


Trial  0 (default) | mean RMSE: 48.1466 ± 7.0216
Trial  1 | mean RMSE: 46.8772 ± 6.9173
Trial  2 | mean RMSE: 49.5417 ± 6.0789
Trial  3 | mean RMSE: 48.2003 ± 5.1984
Trial  4 | mean RMSE: 46.5259 ± 6.0286
Trial  5 | mean RMSE: 46.5933 ± 6.0124
Trial  6 | mean RMSE: 47.5194 ± 6.4060
Trial  7 | mean RMSE: 47.4036 ± 5.8935
Trial  8 | mean RMSE: 48.3754 ± 5.8069
Trial  9 | mean RMSE: 45.8277 ± 5.7977
Trial 10 | mean RMSE: 46.2464 ± 6.2219
Trial 11 | mean RMSE: 62.9193 ± 8.4473
Trial 12 | mean RMSE: 45.4297 ± 5.3938
Trial 13 | mean RMSE: 46.8004 ± 6.7570
Trial 14 | mean RMSE: 45.2387 ± 5.5459
Trial 15 | mean RMSE: 45.2041 ± 5.3767
Trial 16 | mean RMSE: 42.8899 ± 5.1733
Trial 17 | mean RMSE: 49.5605 ± 8.1753
Trial 18 | mean RMSE: 45.3893 ± 4.2329
Trial 19 | mean RMSE: 43.6356 ± 5.7567
Trial 20 | mean RMSE: 42.9203 ± 5.4510

=== Model: LGB ===
Data shape: (300, 55) (n_samples=300, n_features=51)


/Users/zeqing/opt/anaconda3/envs/LA/lib/python3.6/site-packages/sklearn/model_selection/_split.py:668: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  % (min_groups, self.n_splits)), UserWarning)


Trial  0 (default) | mean RMSE: 47.2537 ± 6.1412
Trial  1 | mean RMSE: 46.7940 ± 6.4748
Trial  2 | mean RMSE: 45.8478 ± 5.4330
Trial  3 | mean RMSE: 46.8765 ± 5.8195
Trial  4 | mean RMSE: 45.2678 ± 5.5096
Trial  5 | mean RMSE: 46.0508 ± 5.7806
Trial  6 | mean RMSE: 46.8554 ± 6.6666
Trial  7 | mean RMSE: 46.0519 ± 5.4998
Trial  8 | mean RMSE: 45.6367 ± 6.1341
Trial  9 | mean RMSE: 45.7695 ± 6.4317
Trial 10 | mean RMSE: 47.2374 ± 5.4442
Trial 11 | mean RMSE: 53.4153 ± 6.1623
Trial 12 | mean RMSE: 46.0921 ± 5.7891
Trial 13 | mean RMSE: 46.2362 ± 5.5055
Trial 14 | mean RMSE: 46.5294 ± 6.3680
Trial 15 | mean RMSE: 47.2937 ± 6.5037
Trial 16 | mean RMSE: 47.0071 ± 6.3603
Trial 17 | mean RMSE: 46.1992 ± 6.1660
Trial 18 | mean RMSE: 47.3114 ± 6.5335
Trial 19 | mean RMSE: 48.0404 ± 6.8991
Trial 20 | mean RMSE: 44.5431 ± 5.3812


In [24]:
model_development_results_dict

{'RF': {'trial_results': {0: {'fold_rmses': [34.254057090243585,
     50.45396401844953,
     41.79814188503679,
     58.142162245081096,
     48.97425708122925,
     58.587870629223815,
     40.83719716375826,
     48.11224272608099,
     46.34693437983071,
     42.1625933410371],
    'mean_rmse': 46.966942055997116,
    'std_rmse': 7.258138635199543},
   1: {'fold_rmses': [36.94439257177543,
     51.828908070319144,
     42.52413044994201,
     57.25869156161566,
     48.82348067272863,
     61.14905928071764,
     42.37795297643866,
     46.00086317594875,
     48.27597784691912,
     45.43996966726522],
    'mean_rmse': 48.06234262736702,
    'std_rmse': 6.8437300033504656},
   2: {'fold_rmses': [35.849423955870996,
     51.613189009528085,
     42.4517250037958,
     57.763370556475884,
     49.513892009276404,
     59.81400535604156,
     40.873230022161806,
     45.150097706200505,
     46.825691567640305,
     45.257207154563716],
    'mean_rmse': 47.51118323415551,
    'std_rm